In [7]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch

battery_files = ['B0005.mat', 'B0006.mat', 'B0007.mat', 'B0018.mat']
base_path = '../data/raw/'
window_size = 50

all_windows = []

# 1. Loop through all 4 batteries
for b_file in battery_files:
    mat_data = scipy.io.loadmat(base_path + b_file)
    b_name = b_file.split('.')[0]
    cycles = mat_data[b_name][0, 0]['cycle'][0]
    
    capacities = []
    
    # Extract discharge capacities
    for cycle in cycles:
        if cycle['type'][0] == 'discharge':
            try:
                cap = cycle['data'][0, 0]['Capacity'][0, 0]
                capacities.append(cap)
            except ValueError:
                continue
                
    # 2. Create Sliding Windows
    # If a battery has 168 cycles, this creates 119 samples of length 50
    for i in range(len(capacities) - window_size + 1):
        window = capacities[i : i + window_size]
        all_windows.append(window)

all_windows_np = np.array(all_windows)
print(f"Total training samples generated: {all_windows_np.shape[0]}")
print(f"Shape of each sample: {all_windows_np.shape[1]}")

Total training samples generated: 440
Shape of each sample: 50


In [8]:
# 3. Normalize the data to [-1, 1] for the GAN
scaler = MinMaxScaler(feature_range=(-1, 1))

# We must flatten the array to scale it, then reshape it back
flattened_data = all_windows_np.reshape(-1, 1)
scaled_data = scaler.fit_transform(flattened_data)
scaled_windows_np = scaled_data.reshape(all_windows_np.shape)

# 4. Convert to PyTorch Tensor and Save
training_tensor = torch.tensor(scaled_windows_np, dtype=torch.float32)

# Save this so your models.py can use it without running extraction again
torch.save(training_tensor, '../data/processed/battery_windows.pt')
print("Tensor successfully saved for training!")

Tensor successfully saved for training!


In [6]:
from sklearn.preprocessing import MinMaxScaler
import torch

# Convert list to numpy array and reshape for the scaler
capacities_np = np.array(capacities).reshape(-1, 1)

# Initialize scaler and transform data to [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
capacities_scaled = scaler.fit_transform(capacities_np)

# Convert to PyTorch Tensor
real_data_tensor = torch.tensor(capacities_scaled, dtype=torch.float32)

print(f"Tensor shape: {real_data_tensor.shape}") # Should be roughly [168, 1]

Tensor shape: torch.Size([168, 1])
